In [ ]:
# investigate FEB 233

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import uproot
import sys
import math

dir = "/Users/alexanderantonakis/Desktop/Software/AFrameAnalysis/Macros/"

sys.path.append("../Utils")
sys.path.append("../Configs")

from ChannelMap import ChannelMap

frame = 3
strip_w = 11.2 # cm 

data_file = "../Data/outtree_frame3_16Aug2023.root"
#cluster_file = "../OfficialClusters/clusters_frame1.root"
config = "config_frame3.txt"
run_config = "run_config_frame3.txt"

fig_dir = "../Figs/Diagnostics/Frame"+str(frame)+"/"

# set up the geometry of the frame
map = ChannelMap("../Configs/"+config, "../Configs/"+run_config)
map.initialize_config()
map.initialize_run_config()
map.calculate_params()
print("initialized the geometry and voltages")
print("")

# initialize the horizontal febs --> useful to have
febs = map.mac5
horiz_febs = []
for feb in febs:
    if map.is_horiz(feb):
        horiz_febs.append(feb)
        
print("All Horizontal FEBs in this file:", horiz_febs)




In [ ]:
import ROOT
# Open the ROOT file and the TTree
file = uproot.open(data_file)  # Replace with your ROOT file
tree = file["reco/events"]    

run_data = tree["fRun"].array(library="np")


def get_times(row):
    f = np.array(row["flags"])
    m = np.array(row["mac5"])
    t = np.array(row["timestamp"])
    mask = (f == 3)
    new_t = t[mask]
    new_m = m[mask]
    return new_t, new_m

branches = ["fRun", "flags", "mac5", "timestamp"]  # Add the branches you want

for run in map.runs:
    mask = (run_data == run)
    df = tree.arrays(branches, entry_start=mask.nonzero()[0][0], entry_stop=mask.nonzero()[0][-1] + 1, library="pd")
    events = []
    ratios_238 = []
    ratios_156 = []
    for num in range(0, df.shape[0], 50):
        #print("num", num)
        if num % 4000 == 0:
            print("Event", num)
                    
        event = df.iloc[num]
        t_vals, m_vals = get_times(event)
        #if len(t_vals) < 2000:
        #    continue  

        events.append(num)
        t_vals -= min(t_vals)  
 
        feb238_mask = (np.array(m_vals) == 233)
        feb238_anti_mask = (np.array(m_vals) != 233)
        feb156_mask = (np.array(m_vals) == 166)

        
        t_all = np.array(t_vals)[feb238_anti_mask]
        t_238 = np.array(t_vals)[feb238_mask]
        t_156 = np.array(t_vals)[feb156_mask]

        """
        if len(t_238) > 0:
            if min(t_238) < 30:
                t_238 = [x - 30 if x > 30 else x for x in t_238]
            else:
                t_238 -= 30 # correct the time
        """
        #print("After")
        #print("min t_238", min(t_238))
        #print("max t_238", max(t_238))
        
        t_all = np.concatenate((t_all, t_238))
        #print("t_all", t_all)
    
        time_interval = max(t_all) - min(t_all)
        
        
        dt = 300
        nbins = int(time_interval / dt)
        #print("nbins", nbins)
        
        h = ROOT.TH1D("h", "", nbins, min(t_all), max(t_all))
        
       
        for t in t_all:
            h.Fill(t)
        
  
        Nc = 0
        if h.GetMaximum() < 3:
            print("small number of clusters")
            Nc = 1
            
        for i in range(1, h.GetNbinsX()+1):
            if h.GetBinContent(i) > 3:
                Nc += 1

        ratios_238.append(len(t_238)/Nc)
        ratios_156.append(len(t_156)/Nc)
        h.Delete()
        #h.Reset("ICEM")
            
        #plt.hist(t_slice, bins=nbins, color="b")
        #plt.show()
        #c = ROOT.TCanvas("c", "c", 700, 500)
        #h.SetStats(0)
        #h.GetXaxis().SetRangeUser(min(t_slice)-300, max(t_slice)+300)
        #h.Draw("HIST")
        #h_238.Draw("HIST Same")
        #c.Update()
        #c.Draw()
    
    plt.title("Run"+str(run), fontsize=16)
    plt.scatter(events, ratios_238, label="233")
    plt.scatter(events, ratios_156, label="166")
    plt.xlabel("Event Number", fontsize=14)
    plt.ylabel("FEB Hits / N > 3 Clusters", fontsize=11)
    plt.legend()
    plt.savefig(fig_dir+"frame"+str(frame)+"feb233_rate_issue_run"+str(run)+".png", 
                format='png')
    plt.show()
   

In [ ]:


for run in map.runs:
    mask = (run_data == run)
    df = tree.arrays(branches, entry_start=mask.nonzero()[0][0], entry_stop=mask.nonzero()[0][-1] + 1, library="pd")
    events = []
    ratios_238 = []
    ratios_156 = []
    for num in range(0, df.shape[0], 50):
        #print("num", num)
        if num % 4000 == 0:
            print("Event", num)
                    
        event = df.iloc[num]
        t_vals, m_vals = get_times(event)
 
        events.append(num)
        t_vals -= min(t_vals)  
 
        feb238_mask = (np.array(m_vals) == 233)
        feb238_anti_mask = (np.array(m_vals) != 233)
        feb156_mask = (np.array(m_vals) == 166)

        
        t_all = np.array(t_vals)[feb238_anti_mask]
        t_238 = np.array(t_vals)[feb238_mask]
        t_156 = np.array(t_vals)[feb156_mask]

        """
        if len(t_238) > 0:
            if min(t_238) < 30:
                t_238 = [x - 30 if x > 30 else x for x in t_238]
            else:
                t_238 -= 30 # correct the time

        """
        #print("After")
        #print("min t_238", min(t_238))
        #print("max t_238", max(t_238))
        
        t_all = np.concatenate((t_all, t_238))
        #print("t_all", t_all)
    
        time_interval = max(t_all) - min(t_all)
        
        
        dt = 300
        nbins = int(time_interval / dt)
        
        h = ROOT.TH1D("h", "", nbins, min(t_all), max(t_all))
        h_238 = ROOT.TH1D("h238", "", nbins, min(t_all), max(t_all)) 
        h_156 = ROOT.TH1D("h156", "", nbins, min(t_all), max(t_all)) 
        
        for t in t_all:
            h.Fill(t)

        for t in t_156:
            h_156.Fill(t)

        for t in t_238:
            h_238.Fill(t)
            
        Nc = 0
        if h.GetMaximum() < 3:
            print("small number of clusters")
            Nc = 1

        N238 = 0
        N156 = 0
        for i in range(1, h.GetNbinsX()+1):
            if h.GetBinContent(i) > 5:
                Nc += 1
                if h_238.GetBinContent(i) > 0:
                    N238 += 1
                    
                if h_156.GetBinContent(i) > 0:
                    N156 += 1

        ratios_238.append(N238/Nc)
        ratios_156.append(N156/Nc)
        
        h.Delete()
        h_238.Delete()
        h_156.Delete()
        
        #h.Reset("ICEM")
            
        #plt.hist(t_slice, bins=nbins, color="b")
        #plt.show()
        #c = ROOT.TCanvas("c", "c", 700, 500)
        #h.SetStats(0)
        #h.GetXaxis().SetRangeUser(min(t_slice)-300, max(t_slice)+300)
        #h.Draw("HIST")
        #h_238.Draw("HIST Same")
        #c.Update()
        #c.Draw()
    
    plt.title("Run"+str(run), fontsize=16)
    plt.scatter(events, ratios_238, label="233")
    plt.scatter(events, ratios_156, label="166")
    plt.xlabel("Event Number", fontsize=14)
    plt.ylabel("FEB Hits in cluster / N > 5 Clusters", fontsize=11)
    plt.legend()
    plt.savefig(fig_dir+"frame"+str(frame)+"feb233_cluster_issue_run"+str(run)+".png", 
                format='png')
    plt.show()